In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/nlp/final project/"

'bertopic continue.ipynb'		'sentiment analysis 2.ipynb'
 bertopic_model_sample.pkl		'sentiment analysis.ipynb'
 bertopic_table_sample.csv		 top_companies_overall.csv
 clean_base_dataset.csv			'topic detection bertopic.ipynb'
 df_model_bertopic_topics_org_tech.csv	'topic detection lda.ipynb'
 df_model_bertopic_with_topics.csv	 top_orgs_by_industry.csv
 distilbert_sentiment			 top_orgs_by_topic.csv
'entity extraction.ipynb'		 top_orgs_overall.csv
'final project planning.gdoc'		 top_tech_by_industry.csv
'nlp final project ymao.gslides'	 top_tech_by_topic.csv
'raw data cleaning.ipynb'		 top_tech_overall.csv
'sentiment analysis (1).ipynb'


In [3]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

MODEL_SAVE = "/content/drive/MyDrive/nlp/final project/distilbert_sentiment/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_SAVE)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_SAVE)
model.to(device)
model.eval()

MAX_LEN = 128
LABEL_NAMES = ["negative", "neutral", "positive"]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [4]:
save_path_csv = "/content/drive/MyDrive/nlp/final project/clean_base_dataset.csv"
df = pd.read_csv(save_path_csv)

display(df.head())

,url,date,title,clean_text,text_len,domain
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",3501,blockworks.co
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,5585,boingboing.net
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",5880,boingboing.net
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,4072,citylife.capetown
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,4347,citylife.capetown


In [5]:
# CELL F: Run inference on df (your 200K clean_base_dataset)
# df is already loaded from your earlier cell — no need to reload

# Build inference text: title + first 400 chars of clean_text
df["inference_text"] = (
    df["title"].fillna("") + ". " +
    df["clean_text"].fillna("").str[:400]
)

def predict_sentiment(texts, batch_size=32):
    all_labels, all_probs = [], []
    model.eval()
    for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
        enc = tokenizer(
            texts[i : i + batch_size],
            truncation=True, padding=True,
            max_length=MAX_LEN, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=1).cpu().numpy()
        all_labels.extend([LABEL_NAMES[p] for p in np.argmax(probs, axis=1)])
        all_probs.extend(probs.tolist())
    return all_labels, all_probs

labels, probs = predict_sentiment(df["inference_text"].tolist())

df["sentiment"]          = labels
df["neg_score"]          = [p[0] for p in probs]
df["neu_score"]          = [p[1] for p in probs]
df["pos_score"]          = [p[2] for p in probs]
df["sentiment_compound"] = df["pos_score"] - df["neg_score"]  # -1 to +1

print(df["sentiment"].value_counts())
display(df[["title", "clean_text", "sentiment", "sentiment_compound"]].head())

# Save
out_path = "/content/drive/MyDrive/nlp/final project/articles_with_sentiment.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} articles → {out_path}")

Inference: 100%|██████████| 6134/6134 [12:54<00:00,  7.92it/s]


sentiment
neutral     177884
positive     14832
negative      3560
Name: count, dtype: int64


,title,clean_text,sentiment,sentiment_compound
0,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",neutral,-0.000244
1,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,neutral,0.000151
2,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",neutral,0.000107
3,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,neutral,0.000249
4,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,neutral,0.054185


Saved 196276 articles → /content/drive/MyDrive/nlp/final project/articles_with_sentiment.csv
